# SubspaceResolve: Ideal-Statevector SSH Prototype

This notebook implements a preliminary ideal-statevector reference calculation for recovery of the finite-size near-zero subspace of an open Su-Schrieffer-Heeger chain. It is not a finite-shot, noisy, circuit-level, or quantum-hardware benchmark.


## Imports and output folders

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

results_directory = root / "results"
figures_directory = root / "figures"
results_directory.mkdir(exist_ok=True)
figures_directory.mkdir(exist_ok=True)


## SSH Hamiltonian and exact reference

In [ ]:
def build_ssh_hamiltonian(number_of_sites, intracell_hopping, intercell_hopping):
    if number_of_sites % 2 != 0:
        raise ValueError("The number of sites must be even.")

    hamiltonian = np.zeros((number_of_sites, number_of_sites), dtype=float)

    for site in range(number_of_sites - 1):
        hopping = intracell_hopping if site % 2 == 0 else intercell_hopping
        hamiltonian[site, site + 1] = -hopping
        hamiltonian[site + 1, site] = -hopping

    return hamiltonian


def exact_near_zero_reference(hamiltonian):
    energies, vectors = np.linalg.eigh(hamiltonian)
    target_indices = np.argsort(np.abs(energies))[:2]
    target_energies = energies[target_indices]
    target_vectors = vectors[:, target_indices]

    order = np.argsort(target_energies)
    target_energies = target_energies[order]
    target_vectors = target_vectors[:, order]

    outside_mask = np.ones(len(energies), dtype=bool)
    outside_mask[target_indices] = False
    outside_energies = energies[outside_mask]

    splitting = abs(target_energies[1] - target_energies[0])
    bandwidth = energies[-1] - energies[0]
    lower_gap = np.min(np.abs(outside_energies - target_energies[0]))
    upper_gap = np.min(np.abs(outside_energies - target_energies[1]))
    isolation_gap = min(lower_gap, upper_gap)

    return {
        "all_energies": energies,
        "pair_energies": target_energies,
        "pair_vectors": target_vectors,
        "splitting": splitting,
        "bandwidth": bandwidth,
        "normalized_splitting": splitting / bandwidth,
        "isolation_gap": isolation_gap,
        "normalized_gap": isolation_gap / bandwidth,
    }


## Initial states and variational objectives

In [ ]:
def normalize_state(vector):
    length = np.linalg.norm(vector)
    if length < 1e-14:
        return np.ones_like(vector) / np.sqrt(len(vector))
    return vector / length


def edge_state_guess(number_of_sites, intracell_hopping, intercell_hopping, side):
    ratio = -intracell_hopping / intercell_hopping
    state = np.zeros(number_of_sites, dtype=float)

    if side == "left":
        sites = range(0, number_of_sites, 2)
    elif side == "right":
        sites = range(number_of_sites - 1, -1, -2)
    else:
        raise ValueError("side must be 'left' or 'right'.")

    for power, site in enumerate(sites):
        state[site] = ratio ** power

    return normalize_state(state)


def folded_cost(parameters, squared_hamiltonian):
    state = normalize_state(parameters)
    return float(state @ squared_hamiltonian @ state)


def folded_deflated_cost(parameters, squared_hamiltonian, first_state, penalty):
    state = normalize_state(parameters)
    energy = state @ squared_hamiltonian @ state
    overlap = abs(state @ first_state) ** 2
    return float(energy + penalty * overlap)


## Two-state folded-spectrum recovery

In [ ]:
def optimizer_record(result, state_number, restart_number):
    return {
        "State": state_number,
        "Restart": restart_number,
        "Success": bool(result.success),
        "Status": int(result.status),
        "Message": str(result.message),
        "Iterations": int(result.nit),
        "FunctionEvaluations": int(result.nfev),
        "FinalCost": float(result.fun),
    }


def optimize_state(objective, starting_points, optimizer_options, state_number):
    optimizations = []
    records = []

    for restart_number, starting_point in enumerate(starting_points, start=1):
        result = minimize(
            objective,
            starting_point,
            method="Nelder-Mead",
            options=optimizer_options,
        )
        optimizations.append(result)
        records.append(optimizer_record(result, state_number, restart_number))

    best_index = int(np.argmin([result.fun for result in optimizations]))
    records[best_index]["Selected"] = True

    for index in range(len(records)):
        records[index].setdefault("Selected", False)

    return optimizations[best_index], records


def orthonormalize_two_states(first_state, second_state):
    first_basis_vector = normalize_state(first_state)
    remainder = second_state - (first_basis_vector @ second_state) * first_basis_vector

    if np.linalg.norm(remainder) < 1e-10:
        raise ValueError("The recovered states are nearly parallel.")

    second_basis_vector = normalize_state(remainder)
    return first_basis_vector, second_basis_vector


def recover_projected_pair(hamiltonian, first_basis_vector, second_basis_vector):
    basis = np.column_stack((first_basis_vector, second_basis_vector))
    projected_hamiltonian = basis.T @ hamiltonian @ basis
    internal_energies, internal_vectors = np.linalg.eigh(projected_hamiltonian)

    order = np.argsort(internal_energies)
    internal_energies = internal_energies[order]
    internal_vectors = internal_vectors[:, order]
    recovered_vectors = basis @ internal_vectors

    return {
        "basis": basis,
        "projected_hamiltonian": projected_hamiltonian,
        "internal_energies": internal_energies,
        "internal_vectors": internal_vectors,
        "recovered_vectors": recovered_vectors,
        "splitting": abs(internal_energies[1] - internal_energies[0]),
    }


def run_folded_vqe_two_state(hamiltonian, first_guess, second_guess, seed):
    number_of_sites = hamiltonian.shape[0]
    squared_hamiltonian = hamiltonian @ hamiltonian
    penalty = 2.0 * max(1.0, np.linalg.norm(squared_hamiltonian, 2))
    random_generator = np.random.default_rng(seed)

    optimizer_options = {
        "maxiter": 7000,
        "maxfev": 50000,
        "xatol": 1e-10,
        "fatol": 1e-12,
        "disp": False,
    }

    first_starting_points = [first_guess]
    first_starting_points.extend(random_generator.standard_normal(number_of_sites) for _ in range(3))

    first_objective = lambda parameters: folded_cost(parameters, squared_hamiltonian)
    first_result, first_records = optimize_state(
        first_objective,
        first_starting_points,
        optimizer_options,
        1,
    )
    first_state = normalize_state(first_result.x)

    second_guess = second_guess - (first_state @ second_guess) * first_state
    second_guess = normalize_state(second_guess)
    second_starting_points = [second_guess]
    second_starting_points.extend(
        second_guess + 0.05 * random_generator.standard_normal(number_of_sites)
        for _ in range(3)
    )

    second_objective = lambda parameters: folded_deflated_cost(
        parameters,
        squared_hamiltonian,
        first_state,
        penalty,
    )
    second_result, second_records = optimize_state(
        second_objective,
        second_starting_points,
        optimizer_options,
        2,
    )
    second_state = normalize_state(second_result.x)

    first_basis_vector, second_basis_vector = orthonormalize_two_states(
        first_state,
        second_state,
    )
    recovered = recover_projected_pair(
        hamiltonian,
        first_basis_vector,
        second_basis_vector,
    )

    return {
        "recovered": recovered,
        "first_result": first_result,
        "second_result": second_result,
        "convergence": first_records + second_records,
        "penalty": penalty,
    }


## Subspace fidelity and SSH chain-length sweep

In [ ]:
def subspace_fidelity(exact_vectors, recovered_vectors):
    exact_basis, _ = np.linalg.qr(exact_vectors)
    recovered_basis, _ = np.linalg.qr(recovered_vectors)
    exact_projector = exact_basis @ exact_basis.T
    recovered_projector = recovered_basis @ recovered_basis.T
    fidelity = np.trace(exact_projector @ recovered_projector).real / exact_basis.shape[1]
    return float(np.clip(fidelity, 0.0, 1.0))


intracell_hopping = 0.5
intercell_hopping = 1.0
chain_lengths = [8, 10, 12, 14, 16, 18]

rows = []
all_convergence_records = []

for number_of_sites in chain_lengths:
    hamiltonian = build_ssh_hamiltonian(
        number_of_sites,
        intracell_hopping,
        intercell_hopping,
    )
    exact = exact_near_zero_reference(hamiltonian)
    left_guess = edge_state_guess(
        number_of_sites,
        intracell_hopping,
        intercell_hopping,
        "left",
    )
    right_guess = edge_state_guess(
        number_of_sites,
        intracell_hopping,
        intercell_hopping,
        "right",
    )
    variational = run_folded_vqe_two_state(
        hamiltonian,
        left_guess,
        right_guess,
        200 + number_of_sites,
    )

    recovered = variational["recovered"]
    recovered_energies = recovered["internal_energies"]
    recovered_splitting = recovered["splitting"]
    fidelity = subspace_fidelity(exact["pair_vectors"], recovered["basis"])
    splitting_error = abs(recovered_splitting - exact["splitting"]) / exact["bandwidth"]

    first_result = variational["first_result"]
    second_result = variational["second_result"]

    rows.append(
        {
            "N": number_of_sites,
            "ExactMinus": exact["pair_energies"][0],
            "ExactPlus": exact["pair_energies"][1],
            "RecoveredMinus": recovered_energies[0],
            "RecoveredPlus": recovered_energies[1],
            "ExactDelta": exact["splitting"],
            "RecoveredDelta": recovered_splitting,
            "Bandwidth": exact["bandwidth"],
            "NormalizedDelta": exact["normalized_splitting"],
            "IsolationGap": exact["isolation_gap"],
            "NormalizedGap": exact["normalized_gap"],
            "SubspaceFidelity": fidelity,
            "NormalizedSplittingError": splitting_error,
            "FirstSuccess": bool(first_result.success),
            "FirstStatus": int(first_result.status),
            "FirstMessage": str(first_result.message),
            "FirstIterations": int(first_result.nit),
            "FirstEvaluations": int(first_result.nfev),
            "FirstFinalCost": float(first_result.fun),
            "SecondSuccess": bool(second_result.success),
            "SecondStatus": int(second_result.status),
            "SecondMessage": str(second_result.message),
            "SecondIterations": int(second_result.nit),
            "SecondEvaluations": int(second_result.nfev),
            "SecondFinalCost": float(second_result.fun),
        }
    )

    for record in variational["convergence"]:
        record["N"] = number_of_sites
        all_convergence_records.append(record)

results = pd.DataFrame(rows)
convergence = pd.DataFrame(all_convergence_records)

results.to_csv(results_directory / "ssh_ideal_reference.csv", index=False)

results


## Preliminary ideal-statevector figure

In [ ]:
figure, axis = plt.subplots(figsize=(8.0, 5.2))

axis.plot(
    results["N"],
    results["ExactMinus"],
    color="black",
    marker="o",
    linewidth=1.5,
    label=r"Exact $-\epsilon$",
)
axis.plot(
    results["N"],
    results["ExactPlus"],
    color="red",
    marker="o",
    linewidth=1.5,
    label=r"Exact $+\epsilon$",
)
axis.scatter(
    results["N"],
    results["RecoveredMinus"],
    facecolors="none",
    edgecolors="blue",
    marker="s",
    s=52,
    label=r"Recovered $-\epsilon$",
    zorder=3,
)
axis.scatter(
    results["N"],
    results["RecoveredPlus"],
    facecolors="none",
    edgecolors="teal",
    marker="s",
    s=52,
    label=r"Recovered $+\epsilon$",
    zorder=3,
)

axis.set_xlabel("SSH chain length, N")
axis.set_ylabel("Near-zero eigenvalue")
axis.set_title("Preliminary ideal-statevector SSH near-zero pair")
axis.grid(True, alpha=0.3)
axis.legend()
figure.tight_layout()
figure.savefig(figures_directory / "ssh_ideal_reference.png", dpi=300)
plt.show()


## Compact verification table

In [ ]:
verification_columns = [
    "N",
    "ExactMinus",
    "ExactPlus",
    "RecoveredMinus",
    "RecoveredPlus",
    "SubspaceFidelity",
    "NormalizedSplittingError",
    "FirstSuccess",
    "SecondSuccess",
]

results[verification_columns]
